In [12]:
import numpy as np
import math
from math import sqrt, cos, radians
import pandas as pd

In [13]:
# TEMPERATURE DATA EXTRACTION

# Cardiff
latitude = 51.503
longitude = -3.222

# Newport
# latitude = 51.582
# longitude = -2.898

# Caerphilly
# latitude = 51.650
# longitude = -3.198

# Rhondda Cynon Taf
# latitude = 51.622
# longitude = -3.414

# Vale of Glamorgan
# latitude = 51.448
# longitude = -3.398

# Torfaen
# latitude = 51.698
# longitude = -3.051

# Monmouthshire
# latitude = 51.778
# longitude = -2.903

# Blaenau Gwent
# latitude = 51.754
# longitude = -3.186

# Merthyr Tydfil
#latitude = 51.748
#longitude = -3.364

# Bridgend
# latitude = 51.561
# longitude = -3.614

# Neath Port Talbot
# latitude = 51.645
# longitude = -3.746

# Swansea
#latitude = 51.658
#longitude = -3.967

url = (
    f"https://power.larc.nasa.gov/api/temporal/hourly/point?"
    f"parameters=T2M&community=RE"
    f"&longitude={longitude}"
    f"&latitude={latitude}"
    f"&start=20250101&end=20251231"
    f"&format=csv&header=false"
)

df = pd.read_csv(url)
time_mapping = {"YEAR": "year","MO": "month","DY": "day","HR": "hour"}
df = df.rename(columns=time_mapping)
df["time"] = pd.to_datetime(df[time_mapping.values()])
df = df.drop(columns=time_mapping.values())
df = df.set_index("time")
df = df.resample("30min").interpolate(method="linear")

print(df)

                        T2M
time                       
2025-01-01 00:00:00  10.040
2025-01-01 00:30:00  10.115
2025-01-01 01:00:00  10.190
2025-01-01 01:30:00  10.260
2025-01-01 02:00:00  10.330
...                     ...
2025-12-31 21:00:00  -0.160
2025-12-31 21:30:00  -0.040
2025-12-31 22:00:00   0.080
2025-12-31 22:30:00   0.355
2025-12-31 23:00:00   0.630

[17519 rows x 1 columns]


In [14]:


dt = 1800
dt_hours = dt / 3600  # 0.5 h timestep

common_sched = {
    'sched_people': [
        1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.85,0.4,0.25,0.25,0.25,0.25,0.25,0.25,0.25,
        0.3,0.52,0.87,0.87,0.87,1.0,1.0,1.0
    ],
    'sched_lighting': [
        0.07,0.07,0.07,0.07,0.2,0.4,0.44,0.4,0.18,0.12,0.12,0.12,0.12,0.12,0.12,0.2,
        0.44,0.61,0.82,0.99,1.0,0.7,0.38,0.16
    ],
    'sched_cooking': [
        0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,
        0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0
    ],
    'sched_appliances': [
        0.58,0.45,0.4,0.39,0.38,0.38,0.42,0.54,0.65,0.66,0.67,0.69,0.7,0.69,0.66,0.65,
        0.68,0.81,1.0,0.92,0.89,0.85,0.7,0.7
    ]
}

archetypes = {

    # House Detached

    'Detached_Post1990': {'TFA': 117, 'height': 2.39, 'type': 'detached', 'TMP': 244.94, 'U': {'windows':2.519,'roof':0.224,'floors':0.2,'walls':0.261,'doors':2.89}},

    'Detached_1930_90':  {'TFA': 131.8, 'height': 2.4, 'type': 'detached', 'TMP': 246.24, 'U': {'windows':2.726,'roof':0.465,'floors':0.5,'walls':0.898,'doors':3}},

    'Detached_Pre1930':  {'TFA': 181, 'height': 2.59, 'type': 'detached', 'TMP': 239.73, 'U': {'windows':2.824,'roof':0.868,'floors':0.9,'walls':1.447,'doors':3}},


    # House Semi-detached

    'Semi_Post1990':     {'TFA': 78, 'height': 2.37, 'type': 'semi', 'TMP': 238.63, 'U': {'windows':2.532,'roof':0.239,'floors':0.2,'walls':0.262,'doors':2.89}},

    'Semi_1930_90':  {'TFA': 91, 'height': 2.41, 'type': 'semi', 'TMP': 247.02, 'U': {'windows':2.725,'roof':0.59,'floors':0.5,'walls':1.036,'doors':3}},

    'Semi_Pre1930':  {'TFA': 116, 'height': 2.6, 'type': 'semi', 'TMP': 246.58, 'U': {'windows':2.794,'roof':0.801,'floors':0.9,'walls':1.39,'doors':3}},


    # House Terraced

    'Terraced_Post1990': {'TFA': 76, 'height': 2.37, 'type': 'terraced', 'TMP': 237.76, 'U': {'windows':2.467,'roof':0.238,'floors':0.2,'walls':0.251,'doors':2.89}},

    'Terraced_1930_90':  {'TFA': 86, 'height': 2.4, 'type': 'terraced', 'TMP': 244.69, 'U': {'windows':2.721,'roof':0.499,'floors':0.5,'walls':1.022,'doors':3}},

    'Terraced_Pre1930':  {'TFA': 101, 'height': 2.7, 'type': 'terraced', 'TMP': 246.34, 'U': {'windows':2.747,'roof':0.891,'floors':0.9,'walls':1.641,'doors':3}},


    # Bungalow Detached

    'B_Detached_Post1990': {'TFA': 65, 'height': 2.4, 'type': 'detached', 'TMP': 242.21, 'U': {'windows':2.432,'roof':0.179,'floors':0.2,'walls':0.18,'doors':2.89}},

    'B_Detached_1930_90':  {'TFA': 93, 'height': 2.45, 'type': 'detached', 'TMP': 246.27, 'U': {'windows':2.716,'roof':0.522,'floors':0.5,'walls':0.998,'doors':3}},

    'B_Detached_Pre1930':  {'TFA': 110.7, 'height': 2.58, 'type': 'detached', 'TMP': 239.58, 'U': {'windows':2.841,'roof':0.959,'floors':0.9,'walls':1.369,'doors':3}},


    # Bungalow Semi-detached

    'B_Semi_Post1990':     {'TFA': 60, 'height': 2.38, 'type': 'semi', 'TMP': 238.00, 'U': {'windows':2.459,'roof':0.209,'floors':0.2,'walls':0.3,'doors':2.89}},

    'B_Semi_1930_90':  {'TFA': 63, 'height': 2.4, 'type': 'semi', 'TMP': 246.50, 'U': {'windows':2.707,'roof':0.477,'floors':0.5,'walls':0.914,'doors':3}},

    'B_Semi_Pre1930':  {'TFA': 74, 'height': 2.47, 'type': 'semi', 'TMP': 241.67, 'U': {'windows':2.817,'roof':1.418,'floors':0.9,'walls':1.292,'doors':3}},


    # Bungalow Terraced

    'B_Terraced_Post1990': {'TFA': 59.9, 'height': 2.4, 'type': 'terraced', 'TMP': 234.33, 'U': {'windows':2.516,'roof':0.235,'floors':0.2,'walls':0.36,'doors':2.89}},

    'B_Terraced_1930_90':  {'TFA': 47, 'height': 2.4, 'type': 'terraced', 'TMP': 246.28, 'U': {'windows':2.72,'roof':0.272,'floors':0.5,'walls':0.808,'doors':3}},

    'B_Terraced_Pre1930':  {'TFA': 80.2, 'height': 2.45, 'type': 'terraced', 'TMP': 250.00, 'U': {'windows':2.7,'roof':0.932,'floors':0.9,'walls':1.48,'doors':3}},

}


# Add common schedules
for name, arc in archetypes.items():
    arc.update(common_sched)


# ACH
for name, arc in archetypes.items():

    if 'Post1990' in name:
        arc['ACH'] = 0.6

    elif '1930_90' in name:
        arc['ACH'] = 0.8

    elif 'Pre1930' in name:
        arc['ACH'] = 1.0


# Calculate geometry
for name, arc in archetypes.items():

    TFA    = arc['TFA']
    height = arc['height']
    type_  = arc['type']
    TMP    = arc['TMP']
    ACH = arc['ACH']

    is_bungalow = name.startswith('B_')

    storeys = 1 if is_bungalow else 2

    FA = TFA / storeys
    arc['FA'] = FA

    arc['volume'] = FA * height * storeys

    if type_ == 'detached':
        walls_mult = 4
    elif type_ == 'semi':
        walls_mult = 3
    elif type_ == 'terraced':
        walls_mult = 2
    else:
        walls_mult = 4

    windows_area = TFA * 0.20
    doors_area   = 1.509522 * 2
    perimeter    = 4 * sqrt(FA)
    gross_walls  = perimeter * height * storeys * (walls_mult / 4)
    net_walls    = gross_walls - windows_area - doors_area

    arc['A'] = {
        'windows': windows_area,
        'roof': FA / cos(radians(30)),
        'floors': FA,
        'doors': doors_area,
        'walls': net_walls,
    }

print("Cell 1: Archetype definitions loaded.")

Cell 1: Archetype definitions loaded.


In [15]:
arc_name = 'Semi_Post1990'  # change for next archetype name defined as shown above
suffix   = '_1990HSD'             # change suffix for next archetype

arc = archetypes[arc_name]

FA     = arc['FA']
TFA    = arc['TFA']
height = arc['height']
volume = arc['volume']
TMP    = arc['TMP']
U      = arc['U']
A      = arc['A']
ACH    = arc['ACH']

sched_people     = arc['sched_people']
sched_lighting   = arc['sched_lighting']
sched_cooking    = arc['sched_cooking']
sched_appliances = arc['sched_appliances']


boiler_efficiency = 0.90 #90% boiler efficiency

# No more than 2 degrees C per hour temperature increase for boiler
max_boiler_warmup_rate = 2.0
max_boiler_dt = max_boiler_warmup_rate * dt_hours


# Thermal properties
Cm = TMP * TFA * 1000

Hv = 0.33 * ACH * volume

H_fabric = sum(
    U[c] * A[c]
    for c in U.keys()
)

H_total = H_fabric + Hv
HLP = H_total / TFA

tau = TMP / (3.6 * HLP)
a = 1 + tau / 15

# Number of occupants
N = (1 + 1.76 * (1 - math.exp(-0.000349 * (TFA - 13.9)**2))
     + 0.0013 * (TFA - 13.9))

#Internal gains
Q_people_peak     = N * 60
Q_lighting_peak   = 3.9 * TFA
Q_appliances_peak = 5.4 * TFA
Q_cooking_peak    = 35 + 7 * N


# Temperature set points (HP)
T_set_day   = 20
T_set_night = 18


# Design temperature - Cardiff
T_design_out = -3.1


# Design eqn
Q_design = H_total * (T_set_day - T_design_out)


# Heat pump sizes
hp_sizes = [3500, 5000, 7000, 10000, 12000]

# Boiler sizes
boiler_sizes = [12000, 15000, 18000, 25000, 30000, 35000]


# Heat pump and boiler selection:
def select_capacity(Q_design, available_sizes):

    for size in available_sizes:

        if size >= Q_design:
            return size

    return available_sizes[-1]


Q_hp_max = select_capacity(Q_design, hp_sizes)

Q_boiler_max = select_capacity(Q_design, boiler_sizes)


# Temperature set point (boiler)
T_set_boiler = 20

def boiler_on(hour):
    return (6 <= hour < 9) or (16 <= hour < 22)


avg_24hr = df['T2M'].rolling(48).mean()


def utilisation_factor(G, L, a):

    if abs(L) < 1e-6:
        return 1.0

    gamma = G / L

    if gamma <= 0:
        return 1.0

    elif abs(gamma - 1) < 1e-8:
        return a / (a + 1)

    else:
        return (
            (1 - gamma**a)
            / (1 - gamma**(a + 1))
        )


uf_hp = np.zeros(len(df))
uf_boiler = np.zeros(len(df))

T_hp = np.zeros(len(df))
Q_hp_out = np.zeros(len(df))

T_hp[0] = 18


T_boiler = np.zeros(len(df))
Q_boiler_out = np.zeros(len(df))

T_boiler[0] = 18


HP_RECOVERY_STEPS = 6

hp_recovery_steps = 0
hp_recovery_start_temp = None
hp_recovery_completed = False


for i in range(1, len(df)):

    hour = df.index.hour[i]

    T_out = df['T2M'].iloc[i]


#Heat pump

    Ti = T_hp[i - 1]

    T_set = (
        T_set_day
        if 7 <= hour <= 22
        else T_set_night
    )


    Q_gains_raw = (
        sched_people[hour] * Q_people_peak
        + sched_lighting[hour] * Q_lighting_peak
        + sched_cooking[hour] * Q_cooking_peak
        + sched_appliances[hour] * Q_appliances_peak
    )


    Q_loss = H_total * (Ti - T_out)


    uf = utilisation_factor(Q_gains_raw, Q_loss, a)

    uf_hp[i] = uf

    Q_gains = uf * Q_gains_raw


    summer_cutoff = (
        avg_24hr.iloc[i] > 15
        if not math.isnan(avg_24hr.iloc[i])
        else False
    )


    if summer_cutoff:

        Q_h = 0
        hp_recovery_steps = 0
        hp_recovery_start_temp = None
        hp_recovery_completed = False

    else:

        if 4 <= hour < 7:

            T_target = (T_set_night + (T_set_day - T_set_night) * (hour - 4) / 3)

            Q_needed = (Q_loss - Q_gains+ (Cm / dt) * (T_target - Ti))

            Q_h = max(0, min(Q_hp_max, Q_needed))

            hp_recovery_steps = 0
            hp_recovery_start_temp = None
            hp_recovery_completed = False

        else:

            if hp_recovery_steps == 0:

                if Q_hp_out[i - 1] > 1:

                    hp_recovery_completed = False

                elif Ti < T_set and not hp_recovery_completed:

                    hp_recovery_start_temp = Ti
                    hp_recovery_steps = 1


            if hp_recovery_steps > 0:

                recovery_progress = min(hp_recovery_steps / HP_RECOVERY_STEPS,1.0)
                T_target = (hp_recovery_start_temp + (T_set - hp_recovery_start_temp)* recovery_progress)
                Q_needed = (Q_loss - Q_gains+ (Cm / dt) * (T_target - Ti))
                Q_h = max(0, min(Q_hp_max, Q_needed))

                if hp_recovery_steps >= HP_RECOVERY_STEPS:

                    hp_recovery_steps = 0
                    hp_recovery_start_temp = None
                    hp_recovery_completed = True

                else:

                    hp_recovery_steps += 1


            elif Ti > T_set:

                Q_h = 0


            else:

                Q_needed = (Q_loss - Q_gains+ (Cm / dt) * (T_set - Ti))
                Q_h = max(0, min(Q_hp_max, Q_needed))

    Q_hp_out[i] = Q_h
    T_hp[i] = (Ti + (Q_h + Q_gains - Q_loss) * dt / Cm)

 #boiler

    Ti_b = T_boiler[i - 1]
    Q_loss_b = H_total * (Ti_b - T_out)
    uf_b = utilisation_factor(Q_gains_raw, Q_loss_b, a)
    uf_boiler[i] = uf_b

    Q_gains_b = uf_b * Q_gains_raw

    summer_cutoff_b = (not math.isnan(avg_24hr.iloc[i]) and avg_24hr.iloc[i] > 15)

    if summer_cutoff_b or not boiler_on(hour):

        Q_b = 0

    else:

        dT = T_set_boiler - Ti_b
        Q_needed_b = (Q_loss_b - Q_gains_b + (Cm / dt) * dT)
        Q_b = max(0, min(Q_boiler_max, Q_needed_b))
        Q_b_max_warmup = ((Cm / dt) * max_boiler_dt + Q_loss_b- Q_gains_b)
        Q_b = min(Q_b, Q_b_max_warmup)
        Q_b = max(0, Q_b)

    Q_boiler_out[i] = Q_b
    T_boiler[i] = (Ti_b + (Q_b + Q_gains_b - Q_loss_b) * dt / Cm)

dt_hours = dt/3600

annual_hp_heat_kwh = np.sum(Q_hp_out)*dt_hours/1000
annual_boiler_heat_kwh = np.sum(Q_boiler_out)*dt_hours/1000
print(f"Heat Pump Annual Heat Demand: {annual_hp_heat_kwh:.2f} kWh")
print(f"Boiler Annual Heat Demand: {annual_boiler_heat_kwh:.2f} kWh")


Heat Pump Annual Heat Demand: 5654.41 kWh
Boiler Annual Heat Demand: 5517.40 kWh
